# NB01b — LanceDB Initialisation

**Phase 1, Part B.** Takes the cleaned parquets from NB01 and initialises the LanceDB vector store used by all downstream agents. Creates four tables with BM25 full-text and dense vector indices. Ends with a hybrid-search gate check.

| Table | Source | Rows | Purpose |
|---|---|---|---|
| `restaurants` | restaurants.parquet | 717 | Competitor lookup for market analyst |
| `reviews` | reviews.parquet | 105 774 | Sentiment + retrieval for market analyst |
| `menu_items` | empty schema | 0 | Populated by VLM extraction in Phase 3 |
| `nypl_menus` | NYPL Menu.csv (Italian) | ~74 | Historical style reference for art director |

Run cells top-to-bottom. Each section prints output to inspect before proceeding.

## 1. Environment setup

Imports, project root resolution, and preflight checks. Confirms the venv has `lancedb`, `pyarrow`, and `llama-cpp-python`, and that all source files from NB01 are present before any work starts.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import pyarrow as pa
import lancedb
from tqdm.auto import tqdm
from llama_cpp import Llama

ROOT = Path().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
assert (ROOT / "configs").exists(), f"Cannot find project root from {Path().resolve()}"

PROCESSED = ROOT / "data" / "processed"
RAW       = ROOT / "data" / "raw"
NYPL_DIR  = RAW / "nypl"
DB_PATH   = ROOT / "data" / "lancedb"

assert (PROCESSED / "restaurants.parquet").exists(), "restaurants.parquet missing — run NB01 first"
assert (PROCESSED / "reviews.parquet").exists(),     "reviews.parquet missing — run NB01 first"
assert (NYPL_DIR / "Menu.csv").exists(),             "Menu.csv missing — run NB01 first"

print(f"Project root : {ROOT}")
print(f"lancedb      : {lancedb.__version__}")
print(f"pyarrow      : {pa.__version__}")
print(f"pandas       : {pd.__version__}")

## 2. Configuration

Model path, embedding batch size, and LanceDB destination. All values used downstream come from here — nothing below is hard-coded.

In [ ]:
MODEL_PATH   = ROOT / "models" / "embeddinggemma-300M-BF16.gguf"
N_CTX        = 512   # token context window for embedding model
N_GPU_LAYERS = -1    # all layers on GPU
BATCH_SIZE   = 512   # rows per embedding batch (reviews)

assert MODEL_PATH.exists(), f"Embedding model not found: {MODEL_PATH}"
print(f"Model : {MODEL_PATH.name}")
print(f"DB    : {DB_PATH}")

## 3. Load embedding model and probe output dimension

Loads the 300 M-parameter Gemma embedding model with full GPU offload (`n_gpu_layers=-1`). Embeds a probe string to discover the vector dimension — this value drives the PyArrow schema for every table. Expected VRAM: ~700 MB.

In [ ]:
print("Loading embedding model...")
embedder = Llama(
    model_path=str(MODEL_PATH),
    embedding=True,
    n_ctx=N_CTX,
    n_gpu_layers=N_GPU_LAYERS,
    verbose=False,
)

_probe = embedder.embed("test")
EMBED_DIM = len(_probe)
print(f"Embedding dimension : {EMBED_DIM}")
print(f"Probe vector norm   : {np.linalg.norm(_probe):.4f}")

## 4. Connect to LanceDB

Creates `data/lancedb/` if it does not exist, then opens the connection. Prints any pre-existing tables so a re-run is safe to inspect before deciding whether to proceed.

In [ ]:
DB_PATH.mkdir(parents=True, exist_ok=True)
db = lancedb.connect(str(DB_PATH))

existing = db.table_names()
print(f"DB path        : {DB_PATH}")
print(f"Existing tables: {existing if existing else '(none)'}") 

## 5. Restaurants table — embed, ingest, and index

Reads `restaurants.parquet` (717 rows), embeds each row, and creates the table with both a BM25 FTS index and dense vector support.

**Text embedded:** `name + " " + categories` — captures cuisine type and restaurant identity for both semantic and exact-name retrieval.

**Cell 5A** computes embeddings (fast — 717 × 300 M ≈ 30–60 s). **Cell 5B** creates the table and indices.

In [ ]:
# ── 5A: Load and embed ────────────────────────────────────────────────────────
rdf = pd.read_parquet(PROCESSED / "restaurants.parquet")
print(f"Loaded restaurants.parquet: {rdf.shape}")

texts = (rdf["name"].fillna("") + " " + rdf["categories"].fillna("")).tolist()
vectors = [embedder.embed(t) for t in tqdm(texts, desc="Embedding restaurants")]

rdf["vector"] = vectors
print(f"Embeddings done. Dim={len(vectors[0])} (expected {EMBED_DIM})")

In [ ]:
# ── 5B: Create table, FTS index ───────────────────────────────────────────────
TABLE = "restaurants"
if TABLE in db.table_names():
    db.drop_table(TABLE)
    print(f"Dropped existing '{TABLE}' table.")

tbl_restaurants = db.create_table(TABLE, data=rdf)
tbl_restaurants.create_fts_index("name",       replace=True)
tbl_restaurants.create_fts_index("categories", replace=True)
tbl_restaurants.create_fts_index("address",    replace=True)
# 717 rows — exact vector scan is sufficient; no ANN index needed

print(f"Table '{TABLE}' created")
print(f"  Rows    : {tbl_restaurants.count_rows()}")
print(f"  Columns : {tbl_restaurants.schema.names}")

## 6. Reviews table — schema + FTS index, then batch embed

**Cell 6A** creates an empty table with the full schema (including the vector column) and adds the BM25 FTS index on `text`. This alone enables fast text search for Phase 2 EDA, without waiting for embeddings.

**Cell 6B** processes all 105 774 reviews in batches of `BATCH_SIZE`, embeds each batch, and appends it to the table. At ~300 M parameters this takes roughly 15–25 minutes. **You can run 6B and work on other things** — the FTS index from 6A is fully usable immediately.

In [ ]:
# ── 6A: Create empty table with schema, FTS index ────────────────────────────
revdf = pd.read_parquet(PROCESSED / "reviews.parquet")
revdf["date"] = revdf["date"].dt.strftime("%Y-%m-%d")
print(f"Loaded reviews.parquet: {revdf.shape}")
print(f"Columns: {revdf.columns.tolist()}")

REVIEW_SCHEMA = pa.schema([
    pa.field("review_id",   pa.string()),
    pa.field("business_id", pa.string()),
    pa.field("user_id",     pa.string()),
    pa.field("stars",       pa.float64()),
    pa.field("text",        pa.string()),
    pa.field("date",        pa.string()),
    pa.field("useful",      pa.int64()),
    pa.field("funny",       pa.int64()),
    pa.field("cool",        pa.int64()),
    pa.field("city",        pa.string()),
    pa.field("vector",      pa.list_(pa.float32())),
])

TABLE = "reviews"
if TABLE in db.table_names():
    db.drop_table(TABLE)
    print(f"Dropped existing '{TABLE}' table.")

tbl_reviews = db.create_table(TABLE, schema=REVIEW_SCHEMA)
tbl_reviews.create_fts_index("text", replace=True)
tbl_reviews.create_fts_index("city", replace=True)

print(f"Table '{TABLE}' created (empty, FTS index ready)")
print(f"  Schema: {tbl_reviews.schema.names}")

In [ ]:
# ── 6B: Batch-embed and append rows (long-running ~15–25 min) ─────────────────
n = len(revdf)
print(f"Embedding {n} reviews in batches of {BATCH_SIZE}...")

for start in tqdm(range(0, n, BATCH_SIZE), desc="Embedding reviews", unit="batch"):
    chunk = revdf.iloc[start : start + BATCH_SIZE].copy()
    chunk["vector"] = [embedder.embed(t) for t in chunk["text"].fillna("").tolist()]
    tbl_reviews.add(chunk)

total = tbl_reviews.count_rows()
print(f"Done. Total rows in table: {total}")
assert total == n, f"Row count mismatch: expected {n}, got {total}" 

## 7. menu_items table — Phase 3 schema placeholder

Creates `menu_items` with its final schema and FTS index but no data. Setting the index up now avoids a costly rebuild when the VLM extraction results arrive in Phase 3.

Schema columns match the `MenuDish` Pydantic model Phase 3 will emit: `dish_id`, `business_id`, `dish_name`, `description`, `price`, `section`, `source` (yelp | nypl), `image_path`, `confidence`, `vector`.

In [ ]:
MENU_SCHEMA = pa.schema([
    pa.field("dish_id",     pa.string()),
    pa.field("business_id", pa.string()),
    pa.field("dish_name",   pa.string()),
    pa.field("description", pa.string()),
    pa.field("price",       pa.string()),
    pa.field("section",     pa.string()),
    pa.field("source",      pa.string()),
    pa.field("image_path",  pa.string()),
    pa.field("confidence",  pa.float32()),
    pa.field("vector",      pa.list_(pa.float32())),
])

TABLE = "menu_items"
if TABLE in db.table_names():
    db.drop_table(TABLE)

tbl_menu_items = db.create_table(TABLE, schema=MENU_SCHEMA)
tbl_menu_items.create_fts_index("dish_name",   replace=True)
tbl_menu_items.create_fts_index("description", replace=True)

print(f"Table '{TABLE}' created (empty — populated in Phase 3)")
print(f"  Schema: {tbl_menu_items.schema.names}")
print(f"  Rows  : {tbl_menu_items.count_rows()}") 

## 8. nypl_menus table — full NYPL corpus (17 562 menus)

Reads the complete `Menu.csv` (all 17 562 NYPL menus). No cuisine filter — the art-director agent uses these for layout patterns, section naming conventions, and typographic cues, for which a 1920s French menu and a 1950s Italian menu are equally useful. Matches the plan target of ~200+ style-reference menus.

**Text embedded:** `name + venue + sponsor` — short strings, so 17 k embeddings takes roughly 2–3 minutes.

**Cell 8A** reads and preps. **Cell 8B** embeds and creates the table.

In [ ]:
# ── 8A: Read Menu.csv, prep all rows ────────────────────────────────────────
menus_df = pd.read_csv(NYPL_DIR / "Menu.csv", low_memory=False)
print(f"NYPL Menu.csv: {len(menus_df)} rows, cols: {menus_df.columns.tolist()}")

keep = [c for c in ["id", "name", "sponsor", "event", "venue", "date", "page_count", "dish_count"]
        if c in menus_df.columns]
menus_df = menus_df[keep].rename(columns={"id": "menu_id"})
menus_df["menu_id"] = menus_df["menu_id"].astype(str)
for col in ["name", "sponsor", "event", "venue"]:
    if col in menus_df.columns:
        menus_df[col] = menus_df[col].fillna("")

print(f"Rows to ingest: {len(menus_df)}")
print(menus_df[["menu_id", "name", "venue"]].head(5).to_string())

In [ ]:
# ── 8B: Embed and create nypl_menus table ────────────────────────────────────
embed_texts = (
    menus_df.get("name",    pd.Series([""] * len(menus_df))).fillna("") + " " +
    menus_df.get("venue",   pd.Series([""] * len(menus_df))).fillna("") + " " +
    menus_df.get("sponsor", pd.Series([""] * len(menus_df))).fillna("")
).str.strip().tolist()

menus_df["vector"] = [
    embedder.embed(t) for t in tqdm(embed_texts, desc="Embedding NYPL menus")
]

TABLE = "nypl_menus"
if TABLE in db.table_names():
    db.drop_table(TABLE)

tbl_nypl = db.create_table(TABLE, data=menus_df)
for col in ["name", "venue", "sponsor", "event"]:
    if col in menus_df.columns:
        tbl_nypl.create_fts_index(col, replace=True)

print(f"Table '{TABLE}' created")
print(f"  Rows    : {tbl_nypl.count_rows()}")
print(f"  Columns : {tbl_nypl.schema.names}")

## 9. Gate check — hybrid search

Runs a representative market-analyst query against `restaurants` in three modes: FTS-only (BM25), vector-only (dense cosine), and hybrid (BM25 + dense via Reciprocal Rank Fusion). Gate passes when each mode returns ≥ 1 result. Hybrid is the mode used by the market analyst agent in Phase 4.

In [ ]:
from lancedb.rerankers import RRFReranker

TEST_QUERY = "South Philadelphia Italian casual dining"
query_vec  = embedder.embed(TEST_QUERY)
reranker   = RRFReranker()

print(f"Test query: '{TEST_QUERY}'\n")

# FTS
fts_res = (tbl_restaurants.search(TEST_QUERY, query_type="fts")
           .limit(5).to_pandas())
print(f"FTS-only ({len(fts_res)} results):")
print(fts_res[["name", "city", "stars"]].to_string())

# Vector
vec_res = (tbl_restaurants.search(query_vec, query_type="vector")
           .limit(5).to_pandas())
print(f"\nVector-only ({len(vec_res)} results):")
print(vec_res[["name", "city", "stars"]].to_string())

# Hybrid
hybrid_res = (
    tbl_restaurants.search(query_type="hybrid")
    .vector(query_vec)
    .text(TEST_QUERY)
    .rerank(reranker)
    .limit(5)
    .to_pandas()
)
print(f"\nHybrid ({len(hybrid_res)} results):")
print(hybrid_res[["name", "city", "stars"]].to_string())

gate_ok = len(fts_res) > 0 and len(vec_res) > 0 and len(hybrid_res) > 0
print(f"\nGate: all three modes return ≥1 result  [{'OK' if gate_ok else 'FAIL'}]")

## 10. Stats card — Phase 1b summary

Row counts and on-disk footprint for every table. This is the evidence artefact for the Phase 1b gate check and the horizon scan's data section.

In [ ]:
print("=" * 55)
print("LANCEDB STATS — Phase 1b")
print("=" * 55)

for name in db.table_names():
    t = db.open_table(name)
    rows = t.count_rows()
    lance_dir = DB_PATH / f"{name}.lance"
    size_mb = (
        sum(f.stat().st_size for f in lance_dir.rglob("*") if f.is_file()) / 1e6
        if lance_dir.exists() else 0.0
    )
    print(f"  {name:<20} {rows:>7} rows   {size_mb:>7.1f} MB")

total_mb = sum(f.stat().st_size for f in DB_PATH.rglob("*") if f.is_file()) / 1e6
print(f"  {'TOTAL':<20} {'':>7}        {total_mb:>7.1f} MB")

print()
print("── Phase 1b gate checks ──")
tables = db.table_names()
for expected in ["restaurants", "reviews", "menu_items", "nypl_menus"]:
    ok = expected in tables
    print(f"  Table '{expected:<12}' exists : [{'OK' if ok else 'FAIL'}]")

r_rows = db.open_table("restaurants").count_rows() if "restaurants" in tables else 0
v_rows = db.open_table("reviews").count_rows()     if "reviews"     in tables else 0
print(f"  restaurants rows ≥ 700     : {r_rows}  [{'OK' if r_rows >= 700 else 'FAIL'}]")
print(f"  reviews rows ≥ 100 000     : {v_rows}  [{'OK' if v_rows >= 100_000 else 'FAIL'}]")
print(f"  hybrid search (cell 9)     : [{'OK' if gate_ok else 'FAIL'}]")
print()
print("Next: NB02_eda_restaurants.ipynb")